# Advanced: Multi-Strategy MVF Analysis with PAMOLA.CORE

**Goal**: Master all multi-valued field analysis strategies using operation.execute()

**What you'll learn:**
- **Strategy 1**: Single field detailed analysis (values, combinations, distributions)
- **Strategy 2**: Multiple field comparative analysis (cross-field patterns)
- **Strategy 3**: Large dataset processing (Dask/Joblib parallel)
- Parse complex multi-valued formats
- Analyze value distributions and combinations
- Production deployment patterns

**Prerequisites:**
- Completed the simple notebook
- Understanding of multi-valued fields
- Familiarity with operation.execute() pattern
- Python 3.8+
- PAMOLA.CORE installed (auto-installs if needed)

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── profiling/
        └── 02_mvf_analyzer_advanced.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import json
import sys
import os
from pathlib import Path
from datetime import datetime
import time
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print("   Repository: https://github.com/DGT-Network/PAMOLA.git")
        print("   Branch: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.profiling.analyzers.mvf import MVFOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Complex Dataset

**How to use:**
- Run to load or generate 1000-record dataset with MVF fields
- Auto-creates sample if file not found
- Review data structure before proceeding

**What you'll see:**
- Load status (from file or generated)
- Dataset overview (1000 records, 5 columns)
- Sample records (first 5 rows)
- Column statistics (types, unique counts)

**Note:** Generated data includes realistic multi-valued fields with different formats

In [ ]:
# Try to load from file first
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'mvf_complex_data.csv'
print(f"📂 Looking for data at: {data_path}\n")

if data_path.exists():
    print(f"📂 Loading data from: {data_path}")
    df = read_csv(data_path)
    print(f"✓ Loaded {len(df)} records from file")
else:
    print("📊 Generating synthetic dataset (1000 records)...\n")
    
    np.random.seed(42)
    n_records = 1000
    
    # Generate realistic MVF data
    skills_pool = ['Python', 'Java', 'JavaScript', 'SQL', 'R', 'C++', 'Go', 'Rust',
                   'Machine Learning', 'Data Science', 'Web Development', 'DevOps',
                   'Cloud', 'Docker', 'Kubernetes', 'AWS', 'Azure', 'React', 'Vue',
                   'Django', 'Flask', 'Spring', 'TensorFlow', 'PyTorch']
    
    tools_pool = ['Git', 'Jira', 'Jenkins', 'GitHub', 'GitLab', 'Bitbucket',
                  'VS Code', 'IntelliJ', 'PyCharm', 'Jupyter', 'Postman',
                  'Tableau', 'Power BI', 'Excel', 'MongoDB', 'PostgreSQL']
    
    tags_pool = ['senior', 'junior', 'mid-level', 'lead', 'architect',
                 'full-stack', 'backend', 'frontend', 'data', 'ml', 'ai',
                 'remote', 'hybrid', 'onsite', 'contract', 'permanent']
    
    def generate_mvf_list(pool, min_items=1, max_items=5):
        n_items = np.random.randint(min_items, max_items + 1)
        items = np.random.choice(pool, size=n_items, replace=False)
        return str(list(items))
    
    def generate_mvf_json(pool, min_items=1, max_items=4):
        n_items = np.random.randint(min_items, max_items + 1)
        items = np.random.choice(pool, size=n_items, replace=False)
        return json.dumps(list(items))
    
    df = pd.DataFrame({
        'record_id': range(1, n_records + 1),
        'skills': [generate_mvf_list(skills_pool, 2, 6) for _ in range(n_records)],
        'tools': [generate_mvf_json(tools_pool, 1, 4) for _ in range(n_records)],
        'tags': [generate_mvf_list(tags_pool, 1, 3) for _ in range(n_records)],
        'years_experience': np.random.randint(1, 20, n_records)
    })
    
    try:
        data_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(data_path, index=False)
        print(f"✓ Generated and saved to: {data_path}")
    except PermissionError:
        print(f"⚠️  Cannot save to {data_path} (file may be open)")
        print(f"   Dataset generated in memory only")

print(f"\n📊 Dataset Overview:")
print("=" * 80)
print(f"  Records: {len(df):,}")
print(f"  Columns: {', '.join(df.columns)}")

print(f"\n🔍 Sample Records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:20s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    else:
        print(f"  {col:20s} ({dtype_str:10s}): min={df[col].min()}, max={df[col].max()}")

print(f"\n💡 Perfect dataset for testing all 3 strategies!")

## Step 3: Setup Shared Environment

**How to use:**
1. **CUSTOMIZE FIELD_CONFIG**: Edit field assignments for each strategy
   - `strategy1_field`: Single field for detailed analysis
   - `strategy2_fields`: Multiple fields for comparison
   - `strategy3_field`: Field for large dataset processing
2. Run to validate fields and setup environment

**What you'll see:**
- Field validation (✓ marks for each field)
- Unique value counts per field
- Task directory created (✓)
- TaskReporter initialized (✓)
- DataSource and config ready (✓)

**Note:** All fields must be multi-valued (contain arrays/lists)

In [ ]:
# Field configuration - CUSTOMIZE THESE!
FIELD_CONFIG = {
    "strategy1_field": "skills",            # Single field analysis
    "strategy2_fields": ["skills", "tools", "tags"],  # Multiple field comparison
    "strategy3_field": "skills"             # Large dataset processing
}

# Validate all fields exist
print("📋 Validating field configuration...\n")

all_fields = set()
all_fields.add(FIELD_CONFIG["strategy1_field"])
all_fields.update(FIELD_CONFIG["strategy2_fields"])
all_fields.add(FIELD_CONFIG["strategy3_field"])

for field in sorted(all_fields):
    if field not in df.columns:
        raise ValueError(
            f"❌ Field '{field}' not found!\n"
            f"Available columns: {', '.join(df.columns)}\n"
            f"Please update FIELD_CONFIG"
        )
    print(f"  ✓ {field:20s}: {df[field].nunique()} unique values")

def create_config_kwargs():
    return {"dataset_name": "main_dataset"}

examples_dir = project_root / 'examples'
task_dir = examples_dir / 'data_examples' / 'advanced_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

task_reporter = TaskReporter(
    task_id="advanced_001",
    task_type="multi_strategy",
    description="Multi-strategy MVF analysis",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

kwargs = create_config_kwargs()
print(f"✓ Config kwargs ready")

data_source = DataSource(dataframes={"main_dataset": df})
print("✓ DataSource created")

print(f"\n✅ Shared environment ready for all strategies!")

## STRATEGY 1: Single Field Detailed Analysis

**How to use:**
- Analyzes one multi-valued field comprehensively
- Run to execute detailed MVF profiling
- Monitor progress bar (6 steps)

**Key parameters:**
- `field_name`: MVF field to analyze
- `top_n=20`: Show top 20 values/combinations
- `min_frequency=1`: Include all values in dictionary
- `format_type=None`: Auto-detect format

**What you'll see:**
- Configuration summary
- Progress: validation → parsing → analysis → viz → save
- Completion time and status
- Value/combination statistics

**Note:** Creates comprehensive dictionaries and visualizations

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 1: SINGLE FIELD DETAILED ANALYSIS")
print("=" * 80 + "\n")

tracker_s1 = HierarchicalProgressTracker(
    total=6, description="Strategy 1: Single Field", unit="steps",
    track_memory=True, level=0
)

operation_s1 = MVFOperation(
    field_name=FIELD_CONFIG["strategy1_field"],
    top_n=20,
    min_frequency=1,
    format_type=None,  # Auto-detect
    parse_kwargs={},
    output_format='csv',
    use_vectorization=False,
    use_cache=False,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 1 configured")
print(f"  Field: {operation_s1.field_name}")
print(f"  Top N: {operation_s1.top_n}")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

result_s1 = operation_s1.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy1_single_field',
    reporter=task_reporter,
    progress_tracker=tracker_s1,
    **kwargs
)

elapsed_s1 = time.time() - start_time
print(f"\n✅ Strategy 1 completed in {elapsed_s1:.2f}s")

# Load results
output_dir_s1 = task_dir / 'strategy1_single_field' / 'output'
if output_dir_s1.exists():
    values_files = list(output_dir_s1.glob('*values_dictionary*.csv'))
    combos_files = list(output_dir_s1.glob('*combinations_dictionary*.csv'))
    
    if values_files:
        values_df_s1 = pd.read_csv(values_files[0])
        print(f"\n📊 Values: {len(values_df_s1)} unique values")
    
    if combos_files:
        combos_df_s1 = pd.read_csv(combos_files[0])
        print(f"📊 Combinations: {len(combos_df_s1)} unique combinations")

## STRATEGY 2: Multiple Field Comparative Analysis

**How to use:**
- Analyzes multiple MVF fields for comparison
- Identifies cross-field patterns and complexity
- Run to execute comparative analysis

**Key parameters:**
- Multiple fields analyzed sequentially
- Same configuration for consistency
- Comparative metrics calculated

**What you'll see:**
- Configuration summary per field
- Progress: field 1 → field 2 → field 3 → comparison
- Completion time per field
- Comparative statistics

**Note:** Best for understanding relative complexity across fields

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 2: MULTIPLE FIELD COMPARATIVE ANALYSIS")
print("=" * 80 + "\n")

results_s2 = {}
elapsed_s2_fields = {}

for i, field in enumerate(FIELD_CONFIG["strategy2_fields"], 1):
    print(f"\n📊 Analyzing field {i}/{len(FIELD_CONFIG['strategy2_fields'])}: {field}")
    
    tracker_s2 = HierarchicalProgressTracker(
        total=6, description=f"Strategy 2: Field {i}", unit="steps",
        track_memory=True, level=0
    )
    
    operation_s2 = MVFOperation(
        field_name=field,
        top_n=15,
        min_frequency=1,
        format_type=None,
        parse_kwargs={},
        output_format='csv',
        use_vectorization=False,
        use_cache=False,
        generate_visualization=True,
        save_output=True,
        force_recalculation=False
    )
    
    start_time = time.time()
    
    result = operation_s2.execute(
        data_source=data_source,
        task_dir=task_dir / f'strategy2_field_{i}_{field}',
        reporter=task_reporter,
        progress_tracker=tracker_s2,
        **kwargs
    )
    
    elapsed = time.time() - start_time
    results_s2[field] = result
    elapsed_s2_fields[field] = elapsed
    
    print(f"✅ Field '{field}' completed in {elapsed:.2f}s")

elapsed_s2 = sum(elapsed_s2_fields.values())
print(f"\n✅ Strategy 2 completed in {elapsed_s2:.2f}s total")

# Load comparative stats
print(f"\n📊 Comparative Statistics:")
for field in FIELD_CONFIG["strategy2_fields"]:
    output_dir = task_dir / f'strategy2_field_{FIELD_CONFIG["strategy2_fields"].index(field)+1}_{field}' / 'output'
    if output_dir.exists():
        values_files = list(output_dir.glob('*values_dictionary*.csv'))
        if values_files:
            values_df = pd.read_csv(values_files[0])
            print(f"  {field:15s}: {len(values_df):4d} unique values")

## STRATEGY 3: Large Dataset Processing

**How to use:**
- Processes large datasets with parallel execution
- Uses Dask or Joblib for scalability
- Run to execute optimized analysis

**Key parameters:**
- `use_vectorization=True`: Enable Joblib parallel
- `parallel_processes=4`: Number of workers
- `chunk_size=250`: Records per chunk
- Alternative: `use_dask=True` with `npartitions`

**What you'll see:**
- Configuration summary with parallel settings
- Progress: setup → parallel processing → aggregation
- Completion time and speedup
- Performance comparison

**Note:** Optimal for datasets >10,000 records or complex parsing

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 3: LARGE DATASET PROCESSING")
print("=" * 80 + "\n")

tracker_s3 = HierarchicalProgressTracker(
    total=6, description="Strategy 3: Parallel", unit="steps",
    track_memory=True, level=0
)

# Option 1: Joblib (recommended for most cases)
operation_s3 = MVFOperation(
    field_name=FIELD_CONFIG["strategy3_field"],
    top_n=20,
    min_frequency=1,
    format_type=None,
    parse_kwargs={},
    output_format='csv',
    chunk_size=250,           # Smaller chunks for parallel
    use_vectorization=True,   # Enable Joblib
    parallel_processes=4,     # Number of workers
    use_dask=False,           # Or use Dask instead
    npartitions=None,
    use_cache=False,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 3 configured")
print(f"  Field: {operation_s3.field_name}")
print(f"  Parallel: {'Joblib' if operation_s3.use_vectorization else 'Dask' if operation_s3.use_dask else 'None'}")
print(f"  Workers: {operation_s3.parallel_processes if operation_s3.use_vectorization else operation_s3.npartitions if operation_s3.use_dask else 1}")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

result_s3 = operation_s3.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy3_parallel',
    reporter=task_reporter,
    progress_tracker=tracker_s3,
    **kwargs
)

elapsed_s3 = time.time() - start_time
print(f"\n✅ Strategy 3 completed in {elapsed_s3:.2f}s")

# Compare with Strategy 1 (sequential)
if elapsed_s1 > 0:
    speedup = elapsed_s1 / elapsed_s3
    print(f"\n⚡ Speedup: {speedup:.2f}x faster than sequential processing")

## Step 4: Strategy Comparison

**How to use:**
- Run AFTER all strategies complete
- Review performance and effectiveness

**What you'll see:**
- Execution times (fastest to slowest)
- Total processing time
- Complexity comparison across fields
- Best use case recommendations

**Note:** Parallel processing shows benefits on larger datasets

In [ ]:
print("\n" + "=" * 80)
print("📊 STRATEGY COMPARISON")
print("=" * 80 + "\n")

print("⏱️  Execution Time:")
print(f"  Strategy 1 (Single Field):  {elapsed_s1:6.2f}s")
print(f"  Strategy 2 (Multiple):      {elapsed_s2:6.2f}s")
print(f"  Strategy 3 (Parallel):      {elapsed_s3:6.2f}s")
print(f"  Total:                      {elapsed_s1 + elapsed_s2 + elapsed_s3:6.2f}s")

print(f"\n📈 Field Complexity (Strategy 2):")
for field, elapsed in elapsed_s2_fields.items():
    print(f"  {field:15s}: {elapsed:6.2f}s")

print(f"\n💡 Recommendations:")
print(f"  - Use Strategy 1 for focused single-field analysis")
print(f"  - Use Strategy 2 to compare complexity across fields")
print(f"  - Use Strategy 3 for large datasets (>10k records)")

## Step 5: Detailed Artifact Review

**How to use:**
- Run for comprehensive artifact inspection
- Reviews all strategies automatically
- Displays newest files only

**What you'll see (per strategy):**
1. **Metrics JSON**: MVF statistics, performance data
2. **Output Data**: Value and combination dictionaries
3. **Visualizations**: Distribution charts (latest batch)

**Note:** Only newest files shown to avoid confusion from multiple runs

In [ ]:
strategy_dirs = [
    ('strategy1_single_field', 'Strategy 1: Single Field Analysis'),
    ('strategy2_field_1_skills', 'Strategy 2: Skills Field'),
    ('strategy2_field_2_tools', 'Strategy 2: Tools Field'),
    ('strategy2_field_3_tags', 'Strategy 2: Tags Field'),
    ('strategy3_parallel', 'Strategy 3: Parallel Processing')
]

for dir_name, title in strategy_dirs:
    strategy_dir = task_dir / dir_name
    if not strategy_dir.exists():
        continue
        
    print("\n" + "=" * 80)
    print(f"📊 {title}")
    print("=" * 80)
    
    # 1. Metrics Summary (NEWEST - with filtering)
    metrics_dir = strategy_dir / 'metrics'
    if metrics_dir.exists():
        metrics_files = list(metrics_dir.glob('*.json'))
        if metrics_files:
            filtered = [f for f in metrics_files if not f.name.startswith("data_types")]
            target_files = filtered if filtered else metrics_files
            target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
            
            print(f"\n📄 METRICS: {target_files[0].name}")
            print("   " + "-" * 60)
            
            try:
                with open(target_files[0], 'r') as f:
                    data = json.load(f)
                    metrics = data.get('metrics', {})
                
                # Basic statistics
                print(f"   Field analyzed          : {metrics.get('field_name', 'N/A')}")
                print(f"   Total records           : {metrics.get('total_records', 0):,}")
                print(f"   Null count              : {metrics.get('null_count', 0):,} ({metrics.get('null_percentage', 0):.2f}%)")
                print(f"   Empty arrays            : {metrics.get('empty_arrays_count', 0):,} ({metrics.get('empty_arrays_percentage', 0):.2f}%)")
                
                # Value statistics
                print(f"\n   📊 Value Statistics:")
                print(f"      Unique values        : {metrics.get('unique_values', 0):,}")
                print(f"      Unique combinations  : {metrics.get('unique_combinations', 0):,}")
                print(f"      Avg values/record    : {metrics.get('avg_values_per_record', 0):.2f}")
                print(f"      Max values/record    : {metrics.get('max_values_per_record', 0):,}")
                
                # Top values preview
                if 'values_analysis' in metrics:
                    print(f"\n   🔝 Top 10 Values:")
                    values = metrics['values_analysis']
                    sorted_values = sorted(values.items(), key=lambda x: x[1], reverse=True)[:10]
                    for idx, (value, count) in enumerate(sorted_values, 1):
                        pct = (count / metrics.get('total_records', 1) * 100)
                        print(f"      {idx:2d}. {value:30s}: {count:4,} ({pct:5.2f}%)")
                
                # Top combinations preview
                if 'combinations_analysis' in metrics:
                    print(f"\n   🔗 Top 10 Combinations:")
                    combos = metrics['combinations_analysis']
                    sorted_combos = sorted(combos.items(), key=lambda x: x[1], reverse=True)[:10]
                    for idx, (combo, count) in enumerate(sorted_combos, 1):
                        print(f"      {idx:2d}. {combo:50s}: {count:3,}")
                
                # Value count distribution
                if 'value_counts_distribution' in metrics:
                    print(f"\n   📈 Values Per Record Distribution:")
                    dist = metrics['value_counts_distribution']
                    for count, records in sorted(dist.items(), key=lambda x: int(x[0])):
                        pct = (records / metrics.get('total_records', 1) * 100)
                        print(f"      {count:2s} values: {records:4,} records ({pct:5.2f}%)")
                
            except Exception as e:
                print(f"   ⚠️  Error: {e}")
    
    # 2. Values Dictionary (from output/)
    output_dir = strategy_dir / 'output'
    if output_dir and output_dir.exists():
        values_dict_files = sorted(
            list(output_dir.glob('*_values_dictionary_*.csv')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        
        if values_dict_files:
            print(f"\n📁 VALUES DICTIONARY: {values_dict_files[0].name}")
            print("   " + "-" * 60)
            
            try:
                values_df = pd.read_csv(values_dict_files[0])
                
                if len(values_df) > 0:
                    print(f"   Total unique values     : {len(values_df):,}")
                    
                    # Calculate total frequency
                    total_freq = values_df['frequency'].sum()
                    print(f"   Total occurrences       : {total_freq:,}")
                    
                    # Show top 15 values
                    print(f"\n   Top 15 Values by Frequency:")
                    for idx, row in values_df.head(15).iterrows():
                        value = row['value']
                        freq = row['frequency']
                        pct = row['percentage']
                        print(f"      {idx+1:2d}. {value:30s}: {freq:4,} ({pct:5.2f}%)")
                    
                    if len(values_df) > 15:
                        print(f"\n      ... and {len(values_df) - 15} more values")
                else:
                    print(f"   ℹ️  No values data available")
                    
            except Exception as e:
                print(f"   ⚠️  Error reading values dictionary: {e}")
    
    # 3. Combinations Dictionary (from output/)
    if output_dir and output_dir.exists():
        combos_dict_files = sorted(
            list(output_dir.glob('*_combinations_dictionary_*.csv')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        
        if combos_dict_files:
            print(f"\n🔗 COMBINATIONS DICTIONARY: {combos_dict_files[0].name}")
            print("   " + "-" * 60)
            
            try:
                combos_df = pd.read_csv(combos_dict_files[0])
                
                if len(combos_df) > 0:
                    print(f"   Total unique combinations: {len(combos_df):,}")
                    
                    # Calculate total frequency
                    total_freq = combos_df['frequency'].sum()
                    print(f"   Total occurrences        : {total_freq:,}")
                    
                    # Show top 20 combinations
                    print(f"\n   Top 20 Combinations by Frequency:")
                    for idx, row in combos_df.head(20).iterrows():
                        combo = row['combination']
                        freq = row['frequency']
                        pct = row['percentage']
                        # Truncate long combinations
                        combo_display = combo if len(combo) <= 60 else combo[:57] + "..."
                        print(f"      {idx+1:2d}. {combo_display:60s}: {freq:3,} ({pct:5.2f}%)")
                    
                    if len(combos_df) > 20:
                        print(f"\n      ... and {len(combos_df) - 20} more combinations")
                else:
                    print(f"   ℹ️  No combinations data available")
                    
            except Exception as e:
                print(f"   ⚠️  Error reading combinations dictionary: {e}")
    
    # 4. Visualizations (NEWEST BATCH)
    viz_dir = strategy_dir / 'visualizations'
    if viz_dir.exists():
        viz_files = sorted(
            list(viz_dir.glob('*.png')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            latest_batch = [f for f in viz_files if abs(f.stat().st_mtime - latest_time) < 10]
            
            print(f"\n📊 VISUALIZATIONS: {len(latest_batch)} files")
            for viz in latest_batch[:2]:
                print(f"\n   📈 {viz.stem.replace('_', ' ').title()}")
                try:
                    display(Image(filename=str(viz)))
                except:
                    print(f"      (Display error)")

## Step 6: Export Final Data

**How to use:**
- Run AFTER all strategies complete
- Exports dictionaries and analysis results

**What you'll see:**
- Export confirmation per strategy
- File locations and record counts
- Sample of top values/combinations

**Output structure:**
```
advanced_output/
├── strategy1_single_field/output/
│   ├── *values_dictionary*.csv
│   └── *combinations_dictionary*.csv
├── strategy2_field_1_skills/output/
└── strategy3_parallel/output/
```

**Note:** Dictionaries include frequency and percentage columns

In [ ]:
print("=" * 80)
print("📦 EXPORT SUMMARY")
print("=" * 80)

print(f"\n📂 Export directory: {task_dir}\n")

# Strategy 1
print("Strategy 1: Single Field Analysis")
s1_dir = task_dir / 'strategy1_single_field' / 'output'
if s1_dir.exists():
    values_files = list(s1_dir.glob('*values_dictionary*.csv'))
    combos_files = list(s1_dir.glob('*combinations_dictionary*.csv'))
    print(f"  ✅ Values: {values_files[0].name if values_files else 'N/A'}")
    print(f"  ✅ Combinations: {combos_files[0].name if combos_files else 'N/A'}")
    if values_files:
        df_val = pd.read_csv(values_files[0])
        print(f"     Records: {len(df_val)}")
else:
    print("  ⚠️  Not available")

# Strategy 2 (each field)
print("\nStrategy 2: Multiple Field Comparison")
for i, field in enumerate(FIELD_CONFIG["strategy2_fields"], 1):
    s2_dir = task_dir / f'strategy2_field_{i}_{field}' / 'output'
    if s2_dir.exists():
        values_files = list(s2_dir.glob('*values_dictionary*.csv'))
        print(f"  Field {i} ({field}):")
        if values_files:
            df_val = pd.read_csv(values_files[0])
            print(f"    ✅ {len(df_val)} unique values")

# Strategy 3
print("\nStrategy 3: Parallel Processing")
s3_dir = task_dir / 'strategy3_parallel' / 'output'
if s3_dir.exists():
    values_files = list(s3_dir.glob('*values_dictionary*.csv'))
    if values_files:
        df_val = pd.read_csv(values_files[0])
        print(f"  ✅ Processed {len(df_val)} unique values")
        print(f"  ⚡ Parallel speedup: {elapsed_s1/elapsed_s3:.2f}x")
else:
    print("  ⚠️  Not available")

print("\n" + "=" * 80)
print("✅ EXPORT COMPLETE")
print("=" * 80)
if 'elapsed_s1' in locals():
    print(f"⏱️  Total time: {elapsed_s1 + elapsed_s2 + elapsed_s3:.2f}s")

## 🎯 Summary

**Accomplished:**
- ✅ 3 strategies implemented and compared
- ✅ MVF complexity metrics calculated
- ✅ Performance benchmarking completed
- ✅ Production-ready artifacts generated

**Key takeaways:**
- Single field analysis provides comprehensive value insights
- Multiple field comparison reveals relative complexity
- Parallel processing significantly speeds up large datasets

**Strategy recommendations:**
- **Use Strategy 1** when: Deep dive into specific MVF needed
- **Use Strategy 2** when: Comparing complexity across multiple fields
- **Use Strategy 3** when: Processing large datasets (>10k records)

**Next steps:**
- Test with your own MVF data
- Tune parallel workers for optimal performance
- Combine with other profiling operations

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🐙 [GitHub Repository](https://github.com/DGT-Network/PAMOLA)